# Live slate

Set `TEAM` to an abbreviation on the latest Pinnacle board. Odds come from the newest `pinnacle_nba_*_team.json`. Starters come from the newest Rotowire file that day; a team that has already left the page keeps the last five posted earlier that day.

Layer 1 draws one game environment for that team. Layer 2 is the pregame role for each player on that team and on the opponent: his minutes, his team's form, and the opponent's form. This notebook is the live path, so those windows include every completed game before the slate. The Layer 1 draw is the script. It is not drawn again.

Layer 3 turns that role and that script into minutes. The conditional fit trains on every completed game before the slate, and each window stops before the game being scored. Simulation adds the calibration residual to the raw minute prediction, then allocates the team budget once. The live minutes are the rotation segments that realize that allocation, so the schedule does not rescale it. A segment is five players on the floor for a positive duration. Posted start, who is available, and the margin on that row choose the fives. The durations add up to the script's game minutes, so the team total is five times that length and no player plays longer than that length. A player Rotowire lists as out, or out for the season, is unavailable and gets 0 minutes. Questionable and doubtful players stay available. After the script margin is known, a posted starter is active. Everyone else is active with his close-game rate, or, when the margin is at least 20, with the chance he plays in close games or in garbage time. A script with fewer than seven active players adds the next most likely names. Inactive players get 0 minutes, and the minute budget is split across the players who are in.

Layer 4 turns those minutes and the possession counts into shots and turnovers. The model is fit on the raw logs, not on the rolling rates. The attempt pool follows the team's possessions. Both teams share one timeline, and each Layer 3 segment keeps the events with the five players on the floor. Within that five, the share follows recent attempts per minute. The live profile keeps the latest completed game, and its date has to fall after the fit. Makes and free throws then move so the players' points on that row sum to the nearest whole number of that side's scenario points.

Layer 5 credits the assists, rebounds, steals, and blocks those shots and turnovers already produced. A make can be assisted by a teammate. A miss is split between the two teams, then among the teammates who were on the floor. A steal comes from the opponent's turnovers, and a block comes from an opponent miss. The same windows decide who is credited.

Layer 6 keeps one simulation row as one box score. A prop clears when the stat is strictly above its line. The chance that two props both clear is the share of rows where both do, which is not the product of the two separate chances.

Layer 7 prices points, rebounds, assists, steals, blocks, points plus assists, points plus rebounds plus assists, and rebounds plus assists on that box score against one book. A combo line is the sum of those stats on the same row. `BOOK` starts at PrizePicks and can be set to any other saved book: underdog, novig, prophetx, pinnacle, fanduel, or draftkings. PrizePicks is quoted at -119 on both sides. For the other books the line is the two-way price whose over and under are closest. `prop_over` is the share of rows strictly above the line, and `over_ev` is the profit per dollar on that over. A push returns the stake. The five highest overs are a one-dollar-each portfolio. Its risk is the standard deviation of the combined profit on those same rows.

In [12]:
import argparse
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

def repo_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "nba_game_simulator").is_dir() and (candidate / "src").is_dir():
            return candidate
    raise FileNotFoundError("open this notebook from the nba_quant_analysis repo")

REPO = repo_root()
sys.path.insert(0, str(REPO))
sys.path.insert(0, str(REPO / "nba_game_simulator" / "src"))

from game_enviroment import (
    build_features,
    build_game_table,
    load_gamelogs,
    load_latest_slate,
    load_model,
    print_sim_summary,
    save_model,
    season_of,
    simulate_game,
    train,
)
from lineup import (
    attach_lineup_fill,
    build_lineup_features,
    fill_lineup_gaps,
    merge_lineup_features,
)
from role_context import attach_script, role_for_slate
from game_boxscore import Prop, assemble_box_score, format_outcome, prop_probability
from shared_events import allocate_events, fit_events, live_events, training_events
from game_opportunities import (
    allocate_game_opportunities,
    fit_opportunities,
    live_opportunities,
    training_opportunities,
)
from game_rotations import (
    allocate_rotation,
    fit_rotation,
    invalid_segment_count,
    play_rates,
    script_exposure,
    script_rates,
    segments_from_minutes,
    training_minutes,
)
from src.live.predict_points import lineups_on
from src.pipeline.silver.rotowire import normalize_team_abbreviation
from src.scrapers.arbitrage import canonical_team

TEAM = "OKC"
MODEL_PATH = REPO / "nba_game_simulator" / "game_env_model.json"

In [13]:
log_paths = sorted((REPO / "data" / "silver" / "nba").glob("*/regular_season/player_gamelogs.parquet"))
if not log_paths:
    raise FileNotFoundError("no player gamelogs under data/silver/nba")
roster_path = log_paths[-1]
played = pd.read_parquet(
    roster_path,
    columns=["player_name", "player_id", "team_abbreviation", "game_date"],
)
played["game_date"] = pd.to_datetime(played["game_date"])
roster = (
    played.sort_values(["player_id", "game_date"])
    .groupby("player_id", as_index=False)
    .tail(1)[["player_name", "player_id", "team_abbreviation"]]
)

slate = load_latest_slate(TEAM, roster)
board = json.loads(Path(slate["pinnacle_file"]).read_text())["games"]
posted = lineups_on(slate["date"])

def abbr_for(label):
    wanted = canonical_team(label, "nba")
    for abbr in roster["team_abbreviation"].dropna().unique():
        normalized = normalize_team_abbreviation(abbr)
        if canonical_team(normalized, "nba") == wanted:
            return normalized
    return label

rows = []
for game in board:
    participants = game.get("participants") or []
    if len(participants) < 2:
        continue
    away_name, home_name = participants[0], participants[1]
    markets = game.get("team_markets") or {}
    total = None
    for block in markets.get("total") or []:
        if block.get("period") not in (0, None) or block.get("is_alternate"):
            continue
        for line in block.get("lines") or []:
            if line.get("side") == "over" and line.get("points") is not None:
                total = line["points"]
                break
    home_spread = None
    for block in markets.get("spread") or []:
        if block.get("period") not in (0, None) or block.get("is_alternate"):
            continue
        for line in block.get("lines") or []:
            if line.get("side") == "home" and line.get("points") is not None:
                home_spread = line["points"]
                break
    home_abbr = abbr_for(home_name)
    away_abbr = abbr_for(away_name)
    rows.append(
        {
            "away": away_abbr,
            "home": home_abbr,
            "home_spread": home_spread,
            "total": total,
            "away_five": " ".join(posted.get(away_abbr) or []) or "—",
            "home_five": " ".join(posted.get(home_abbr) or []) or "—",
        }
    )

print(f"Pinnacle: {Path(slate['pinnacle_file']).name}")
print(f"Rotowire: {Path(slate['lineup_file']).name}")
print(f"roster:   {roster_path.relative_to(REPO)} through {played['game_date'].max().date()}")
display(pd.DataFrame(rows))

print(
    f"{slate['team']} vs {slate['opponent']}"
    f"  home={slate['is_home']}  spread={slate['spread']}  total={slate['total']}"
)
if slate["team_lineup"] is None or slate["opponent_lineup"] is None:
    print("A five is missing from today's Rotowire files. That side uses the training-average starters.")
else:
    print(f"starter ids  {slate['team']}: {slate['team_lineup']}")
    print(f"starter ids  {slate['opponent']}: {slate['opponent_lineup']}")
for side, missing in (
    (slate["team"], slate["team_out_absent"]),
    (slate["opponent"], slate["opponent_out_absent"]),
):
    if missing:
        print(
            f"{side} listed out with no game on this roster, so no minutes:"
            f" {', '.join(missing)}"
        )

Pinnacle: pinnacle_nba_2026-11-07_111500_team.json
Rotowire: rotowire_nba_2026-11-07_111500.json
roster:   data/silver/nba/2026-27/regular_season/player_gamelogs.parquet through 2026-11-06


,away,home,home_spread,total,away_five,home_five
0,BKN,ATL,-7.0,227.0,Nolan Traore Michael Porter Chaney Johnson Jul...,Kingston Flemings Nickeil Alexander-Walker Dys...
1,DEN,IND,8.0,229.0,Jamal Murray Christian Braun Cameron Johnson A...,Andrew Nembhard Kelly Oubre Aaron Nesmith Pasc...
2,HOU,PHX,6.0,224.5,Fred VanVleet Amen Thompson Kevin Durant Jabar...,Devin Booker Jalen Green Dillon Brooks Miles B...
3,NYK,CLE,5.0,223.0,Jalen Brunson Mikal Bridges Landry Shamet OG A...,James Harden Donovan Mitchell Meleek Thomas Pe...
4,ORL,GSW,-5.5,222.0,Jase Richardson Anthony Black Desmond Bane Fra...,Stephen Curry Brandin Podziemski Yaxel Lendebo...
5,SAS,POR,5.5,229.0,De'Aaron Fox Stephon Castle Dylan Harper Devin...,Damian Lillard Scoot Henderson Toumani Camara ...
6,WAS,OKC,-12.0,222.5,Trae Young Kyshawn George AJ Dybantsa Anthony ...,Shai Gilgeous-Alexander Ajay Mitchell Alex Car...


OKC vs WAS  home=True  spread=-12.0  total=222.5
starter ids  OKC: [1628983, 1642349, 1627936, 1629026, 1628392]
starter ids  WAS: [1629027, 1642273, 1643407, 203076, 1642259]
OKC listed out with no game on this roster, so no minutes: Thomas Sorber
WAS listed out with no game on this roster, so no minutes: Tristan Vukcevic, Jamir Watkins


In [14]:
saved = json.loads(MODEL_PATH.read_text()) if MODEL_PATH.exists() else None
if saved is not None and saved.get("version") == 3:
    model = load_model(MODEL_PATH)
    print(f"loaded {MODEL_PATH.name}  ({model.get('trained_games')} games)")
else:
    if saved is not None:
        print(
            f"{MODEL_PATH.name} is version {saved.get('version')}; "
            "retraining version 3"
        )
    raw = load_gamelogs(str(REPO / "data" / "silver" / "nba" / "*" / "regular_season" / "player_gamelogs.parquet"))
    games = build_game_table(raw)
    lineups, lineup_state = build_lineup_features(
        raw, halflife=12.0, prior_weight=4.0, season_carry=0.6, window=300,
    )
    games = merge_lineup_features(games, lineups)
    lineup_state = attach_lineup_fill(lineup_state, games)
    games = fill_lineup_gaps(games)
    games, state = build_features(
        games, halflife=12.0, prior_w=4.0, carry=0.6, window=300, min_league=30,
    )
    state["lineup"] = lineup_state
    args = argparse.Namespace(
        test_seasons=1,
        validation_season="2025-26",
        holdout_season="2026-27",
        halflife=12.0,
        prior_weight=4.0,
        season_carry=0.6,
        league_window=300,
        spread_convention="auto",
        plots=False,
    )
    model = train(games, state, args)
    save_model(model, str(MODEL_PATH))

sim = simulate_game(
    model,
    slate["team"],
    slate["opponent"],
    slate["date"],
    total=slate["total"],
    spread=slate["spread"],
    is_home=slate["is_home"],
    team_lineup=slate["team_lineup"],
    opponent_lineup=slate["opponent_lineup"],
    n=5000,
    seed=7,
)
print_sim_summary(sim, slate["total"], slate["spread"])
sim[["team_poss", "opp_poss", "pace", "team_pts", "opp_pts", "game_total", "team_margin", "n_ot"]].describe().round(2)

loaded game_env_model.json  (8367 games)

WAS at OKC | total 222.5 | team spread -12.0
               mean     sd     p05     p25     p50     p75     p95
pace          99.88   3.93   93.42   97.34   99.83  102.48  106.41
team_poss    100.29   4.65   93.34   97.41   99.94  102.74  107.98
team_pts     117.47  11.36   98.71  109.91  117.57  125.35  136.13
opp_pts      105.45  11.45   86.65   97.70  105.31  113.09  124.04
game_total   222.92  18.16  193.08  210.69  223.29  235.24  251.81
team_margin   12.03  13.81  -10.94    2.94   11.94   21.48   34.58
P(OT): 3.6%


,team_poss,opp_poss,pace,team_pts,opp_pts,game_total,team_margin,n_ot
count,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00
mean,100.29,100.32,99.88,117.47,105.45,222.92,12.03,0.04
std,4.65,4.64,3.93,11.36,11.45,18.16,13.81,0.23
min,84.85,84.85,84.85,72.69,67.05,156.40,-39.50,0.00
25%,97.41,97.43,97.34,109.91,97.70,210.69,2.94,0.00
50%,99.94,100.02,99.83,117.57,105.31,223.29,11.94,0.00
75%,102.74,102.82,102.48,125.35,113.09,235.24,21.48,0.00
max,133.78,134.38,114.29,161.47,152.15,287.45,63.30,4.00


In [15]:
ROLE_COLUMNS = [
    "season_year",
    "player_id",
    "player_name",
    "team_id",
    "team_abbreviation",
    "game_id",
    "game_date",
    "matchup",
    "min",
    "pf",
    "team_pace",
    "team_off_rating",
    "team_def_rating",
    "pace",
    "off_rating",
    "def_rating",
    "start_position",
    "team_pts",
    "opp_pts",
    "player_team_spread",
    "fga",
    "fgm",
    "fg3_a",
    "fg3_m",
    "fta",
    "ftm",
    "tov",
    "team_fga",
    "team_poss",
    "team_tov",
    "ast",
    "oreb",
    "dreb",
    "stl",
    "blk",
    "team_fgm",
    "team_ast",
    "team_oreb",
    "team_stl",
    "team_blk",
    "opp_dreb",
    "opp_tov",
    "opp_fga",
]
logs = pd.concat(
    [pd.read_parquet(path, columns=ROLE_COLUMNS) for path in log_paths],
    ignore_index=True,
)
logs["game_date"] = pd.to_datetime(logs["game_date"])
slate_day = pd.Timestamp(slate["date"])
history = logs.loc[logs["game_date"] < slate_day]
if history.empty:
    raise ValueError(f"no completed games before {slate['date']}")
# Who is on the team comes from this season. The windows below still use earlier seasons.
this_season = history.loc[history["season_year"].astype(str).eq(season_of(slate_day))]
latest = (
    this_season.sort_values(["player_id", "game_date", "game_id"])
    .groupby("player_id", as_index=False)
    .tail(1)
)
latest["abbr"] = latest["team_abbreviation"].map(normalize_team_abbreviation)


def team_id_for(abbr: str) -> int:
    ids = latest.loc[latest["abbr"].eq(abbr), "team_id"].dropna()
    if ids.empty:
        raise ValueError(f"no completed games for {abbr} before {slate['date']}")
    return int(ids.mode().iloc[0])


def players_for(
    abbr, own_id, other_id, is_home, spread, starter_ids, out_ids, probable_ids
):
    """Players last seen on this team, plus anyone in the posted five.

    ``out_ids`` are Rotowire out and out-for-season players. They stay on the
    roster with ``available`` 0, so the segment schedule gives them 0 minutes.
    Probable players stay available. Posted starters are active in every script.
    """
    starters = {int(player_id) for player_id in (starter_ids or [])}
    outs = {int(player_id) for player_id in (out_ids or [])}
    probable = {int(player_id) for player_id in (probable_ids or [])}
    side = latest.loc[latest["abbr"].eq(abbr)]
    posted = latest.loc[
        latest["player_id"].astype(int).isin(starters | outs | probable)
    ]
    missing = (starters | outs | probable).difference(posted["player_id"].astype(int))
    if missing:
        print(f"{abbr} posted players missing from logs: {sorted(missing)}")
    side = pd.concat([side, posted], ignore_index=True).drop_duplicates("player_id")
    if side.empty:
        raise ValueError(f"no players for {abbr} before {slate['date']}")
    on_roster = side["player_id"].astype(int)
    return pd.DataFrame(
        {
            "player_id": side["player_id"].to_numpy(),
            "player_name": side["player_name"].to_numpy(),
            "team": abbr,
            "game_date": slate_day,
            "team_id": own_id,
            "opp_team_id": other_id,
            "is_home": float(bool(is_home)),
            "is_starter": on_roster.isin(starters).astype(float).to_numpy(),
            "probable": on_roster.isin(probable).astype(float).to_numpy(),
            "available": (~on_roster.isin(outs)).astype(float).to_numpy(),
            "team_spread": float(spread),
        }
    )


team_abbr = normalize_team_abbreviation(slate["team"])
opp_abbr = normalize_team_abbreviation(slate["opponent"])
team_id = team_id_for(team_abbr)
opp_id = team_id_for(opp_abbr)
players = pd.concat(
    [
        players_for(
            team_abbr,
            team_id,
            opp_id,
            slate["is_home"],
            slate["spread"],
            slate["team_lineup"],
            slate["team_out"],
            slate["team_probable"],
        ),
        players_for(
            opp_abbr,
            opp_id,
            team_id,
            not slate["is_home"],
            -float(slate["spread"]),
            slate["opponent_lineup"],
            slate["opponent_out"],
            slate["opponent_probable"],
        ),
    ],
    ignore_index=True,
)
names = players[["player_id", "player_name", "team", "available"]]
role = role_for_slate(logs, players.drop(columns=["player_name", "team"]))
role = names.merge(role, on="player_id", how="left", validate="one_to_one")
role = role.merge(
    players[["player_id", "probable"]],
    on="player_id",
    how="left",
    validate="one_to_one",
)
role["side"] = role["team"].eq(team_abbr).map({True: 0, False: 1})
role = role.sort_values(
    ["side", "is_starter", "min_mean_10"],
    ascending=[True, False, False],
).drop(columns="side")
rates = play_rates(history, role)
role["play_rate"] = rates["play_rate"].to_numpy()
recent_ids = []
for side_id in role["team_id"].drop_duplicates():
    side_games = (
        history.loc[history["team_id"].eq(side_id), ["game_id", "game_date"]]
        .drop_duplicates("game_id")
        .sort_values("game_date")
        .tail(12)["game_id"]
    )
    recent_ids.extend(side_games.astype(str))
recent_ids = set(recent_ids)
pbp = pd.read_parquet(REPO / "data" / "PlayByPlay" / "s27_pbp.parquet")
pbp = pbp.loc[pbp["gameId"].astype(str).isin(recent_ids)]
exposure_logs = history.loc[
    history["game_id"].astype(str).isin(recent_ids),
    ["game_id", "player_id", "team_id", "game_date", "min", "player_name", "start_position"],
]
script = script_rates(script_exposure(pbp, exposure_logs), role)
role["close_rate"] = script["close_rate"].to_numpy()
role["blowout_rate"] = script["blowout_rate"].to_numpy()

shown = [
    "team",
    "player_name",
    "is_starter",
    "available",
    "min_mean_10",
    "play_rate",
    "close_rate",
    "blowout_rate",
    "min_std_10",
    "start_rate_10",
    "pf_mean_10",
    "season_games_prior",
    "team_pace_10",
    "team_off_10",
    "team_def_10",
    "team_rest",
    "team_b2b",
    "opp_pace_10",
    "opp_off_10",
    "opp_def_10",
    "opp_rest",
    "opp_b2b",
    "team_spread",
]
print(
    f"live role through {history['game_date'].max().date()} for {slate_day.date()}"
    f"  ({len(role)} players, latest game included)"
)
out_now = role.loc[role["available"].eq(0), ["team", "player_name"]]
if out_now.empty:
    print("Rotowire out: none on these rosters")
else:
    listed = ", ".join(
        f"{row.team} {row.player_name}" for row in out_now.itertuples(index=False)
    )
    print(f"Rotowire out: {listed}")
display(role[shown].round(2))

team_role = role.loc[role["team"].eq(team_abbr)].reset_index(drop=True)
opp_role = role.loc[role["team"].eq(opp_abbr)].reset_index(drop=True)
opp_env = sim.copy()
opp_env["team_margin"] = -opp_env["team_margin"]
opp_env["is_home"] = 1.0 - opp_env["is_home"].astype(float)
attached = {
    team_abbr: attach_script(team_role, sim),
    opp_abbr: attach_script(opp_role, opp_env),
}
for label, frame in attached.items():
    draws = frame.drop_duplicates("sim_id")
    one = frame.loc[frame["sim_id"].eq(0), "team_final_margin"]
    if one.nunique() != 1:
        raise AssertionError(f"{label} players do not share one margin")
    print(
        f"{label}: {frame['player_id'].nunique()} players share the draw"
        f"  mean margin {draws['team_final_margin'].mean():.1f}"
        f"  P(OT) {draws['went_ot'].mean():.1%}"
        f"  P(blowout) {draws['blowout'].mean():.1%}"
    )


live role through 2026-11-06 for 2026-11-07  (32 players, latest game included)
Rotowire out: OKC Jalen Williams, OKC Chet Holmgren, OKC Cason Wallace, OKC Jaylin Williams, WAS Justin Champagnie, WAS Reece Beekman


,team,player_name,is_starter,available,min_mean_10,play_rate,close_rate,blowout_rate,min_std_10,start_rate_10,...,team_off_10,team_def_10,team_rest,team_b2b,opp_pace_10,opp_off_10,opp_def_10,opp_rest,opp_b2b,team_spread
2,OKC,Shai Gilgeous-Alexander,1.0,1.0,32.97,0.81,0.89,0.67,3.51,1.00,...,112.08,111.84,1.0,1.0,99.90,112.18,113.91,1.0,1.0,-12.0
10,OKC,Ajay Mitchell,1.0,1.0,27.51,0.81,0.89,0.67,4.07,0.60,...,112.08,111.84,1.0,1.0,99.90,112.18,113.91,1.0,1.0,-12.0
1,OKC,Isaiah Hartenstein,1.0,1.0,24.23,0.81,0.89,0.50,3.36,1.00,...,112.08,111.84,1.0,1.0,99.90,112.18,113.91,1.0,1.0,-12.0
3,OKC,Kenrich Williams,1.0,1.0,19.43,0.88,0.89,0.50,10.17,0.60,...,112.08,111.84,1.0,1.0,99.90,112.18,113.91,1.0,1.0,-12.0
0,OKC,Alex Caruso,1.0,1.0,17.97,0.59,0.67,0.50,6.16,0.00,...,112.08,111.84,1.0,1.0,99.90,112.18,113.91,1.0,1.0,-12.0
5,OKC,Jalen Williams,0.0,0.0,27.53,0.38,0.44,0.33,4.35,1.00,...,112.08,111.84,1.0,1.0,99.90,112.18,113.91,1.0,1.0,-12.0
4,OKC,Chet Holmgren,0.0,0.0,27.11,0.38,0.44,0.33,5.02,1.00,...,112.08,111.84,1.0,1.0,99.90,112.18,113.91,1.0,1.0,-12.0
9,OKC,Jared McCain,0.0,1.0,25.32,0.88,0.89,0.50,4.01,0.50,...,112.08,111.84,1.0,1.0,99.90,112.18,113.91,1.0,1.0,-12.0
11,OKC,Micah Peavy,0.0,1.0,25.20,0.81,0.60,0.75,11.66,0.30,...,112.08,111.84,1.0,1.0,99.90,112.18,113.91,1.0,1.0,-12.0
8,OKC,Cason Wallace,0.0,0.0,20.80,0.38,0.44,0.33,6.76,0.50,...,112.08,111.84,1.0,1.0,99.90,112.18,113.91,1.0,1.0,-12.0


OKC: 16 players share the draw  mean margin 12.0  P(OT) 3.6%  P(blowout) 29.5%
WAS: 16 players share the draw  mean margin -12.0  P(OT) 3.6%  P(blowout) 29.5%


In [16]:
trained = training_minutes(history)
rotation = fit_rotation(trained)
print(
    f"rotation fit on {len(trained):,} player-games"
    f" through {history['game_date'].max().date()}"
    "  (each window stops before that game)"
)

game_key = f"{slate_day.date()}-{team_abbr}-{opp_abbr}"
for frame in (sim, opp_env):
    frame["game_id"] = game_key
    frame["game_date"] = slate_day

rng = np.random.default_rng(7)
minutes = {
    team_abbr: allocate_rotation(
        rotation, team_role, sim, rng, method="segments"
    ),
    opp_abbr: allocate_rotation(
        rotation, opp_role, opp_env, rng, method="segments"
    ),
}
schedules = {
    team_abbr: segments_from_minutes(team_role, sim, minutes[team_abbr]),
    opp_abbr: segments_from_minutes(opp_role, opp_env, minutes[opp_abbr]),
}


def script_zero(label, side, environment, drawn):
    """One attached script, realized from that row's allocated minutes."""
    row = environment.iloc[[0]]
    game_minutes = float(row["game_minutes"].iloc[0])
    n_ot = float(row["n_ot"].iloc[0])
    margin = float(row["team_margin"].iloc[0])
    segments = schedules[label][0]
    invalid = invalid_segment_count(
        segments,
        side,
        game_minutes=game_minutes,
        n_ot=n_ot,
    )
    if invalid != 0:
        raise AssertionError(f"{label} script 0 has {invalid} invalid segments")
    total = float(drawn[0].sum())
    if abs(total - 5.0 * game_minutes) > 1e-4:
        raise AssertionError(f"{label} script 0 misses the team minute budget")
    print(
        f"{label} script 0: n_ot {int(round(n_ot))}  "
        f"team_margin {margin:+.1f}  "
        f"cap {game_minutes:.1f}  "
        f"team total {total:.1f}  "
        f"invalid segments {invalid}"
    )


script_zero(team_abbr, team_role, sim, minutes[team_abbr])
script_zero(opp_abbr, opp_role, opp_env, minutes[opp_abbr])


def minute_table(label, side, drawn):
    budget = 5.0 * sim["game_minutes"].to_numpy()
    gap = np.max(np.abs(drawn.sum(axis=1) - budget))
    if gap > 1e-4:
        raise AssertionError(f"{label} minutes miss the script budget by {gap:.4f}")
    cap = sim["game_minutes"].to_numpy()[:, None]
    if np.any(drawn > cap + 1e-6):
        raise AssertionError(f"{label} has a player above the game length")
    summary = pd.DataFrame(
        {
            "team": label,
            "player_name": side["player_name"].to_numpy(),
            "is_starter": side["is_starter"].to_numpy(),
            "min_mean_10": side["min_mean_10"].to_numpy(),
            "mean": drawn.mean(axis=0),
            "sd": drawn.std(axis=0),
            "p10": np.percentile(drawn, 10, axis=0),
            "p90": np.percentile(drawn, 90, axis=0),
        }
    )
    return summary.sort_values(["is_starter", "mean"], ascending=[False, False])


shown_minutes = pd.concat(
    [
        minute_table(team_abbr, team_role, minutes[team_abbr]),
        minute_table(opp_abbr, opp_role, minutes[opp_abbr]),
    ],
    ignore_index=True,
)
team_means = shown_minutes.groupby("team")["mean"].sum().round(1)
print(
    f"segment minutes for {slate_day.date()}  "
    f"mean team total {team_means.to_dict()}"
)
print("sd is the spread across scripts; one script has a fixed schedule")
display(shown_minutes.round(1))

rotation fit on 179,719 player-games through 2026-11-06  (each window stops before that game)
OKC script 0: n_ot 0  team_margin +3.1  cap 48.0  team total 240.0  invalid segments 0
WAS script 0: n_ot 0  team_margin -3.1  cap 48.0  team total 240.0  invalid segments 0
segment minutes for 2026-11-07  mean team total {'OKC': 241.0, 'WAS': 241.0}
sd is the spread across scripts; one script has a fixed schedule


,team,player_name,is_starter,min_mean_10,mean,sd,p10,p90
0,OKC,Shai Gilgeous-Alexander,1.0,33.0,36.3,7.0,28.0,45.5
1,OKC,Ajay Mitchell,1.0,27.5,34.2,7.3,25.1,43.8
2,OKC,Alex Caruso,1.0,18.0,29.1,7.8,19.2,38.9
3,OKC,Isaiah Hartenstein,1.0,24.2,28.1,7.2,19.2,37.1
4,OKC,Kenrich Williams,1.0,19.4,25.8,7.7,16.1,35.5
5,OKC,Jared McCain,0.0,25.3,20.5,9.2,1.1,30.2
6,OKC,Aday Mara,0.0,18.9,16.9,8.7,1.8,27.5
7,OKC,Micah Peavy,0.0,25.2,15.6,11.8,0.0,28.9
8,OKC,Bennett Stirtz,0.0,16.8,12.7,9.8,0.0,25.0
9,OKC,Josh Dix,0.0,13.6,12.2,8.6,0.0,23.7


In [17]:
trained_shots = training_opportunities(history)
shot_model = fit_opportunities(history)
shot_profile = live_opportunities(
    history, players.drop(columns=["player_name", "team"])
)
shot_profile["player_id"] = shot_profile["player_id"].astype(
    players["player_id"].dtype
)
shot_profile = names.drop(columns="available").merge(
    shot_profile, on="player_id", how="left", validate="one_to_one"
)
print(
    f"shot model fit on {shot_model.metadata['player_games']:,} player-games"
    f" through {shot_model.training_cutoff}"
    f"  ({len(trained_shots):,} rolling rows; each window stops before that game)"
)


def side_profile(side):
    ordered = (
        shot_profile.set_index("player_id").loc[side["player_id"]].reset_index()
    )
    if len(ordered) != len(side) or not ordered["player_id"].equals(side["player_id"]):
        raise AssertionError("shot profile does not match the rotation")
    return ordered


if sim["is_home"].nunique() != 1:
    raise AssertionError("the slate does not have one home flag")
slate_is_home = bool(sim["is_home"].iloc[0])
home_abbr, away_abbr = (
    (team_abbr, opp_abbr) if slate_is_home else (opp_abbr, team_abbr)
)
home_role, away_role = (
    (team_role, opp_role) if slate_is_home else (opp_role, team_role)
)
allocated = allocate_game_opportunities(
    shot_model,
    side_profile(home_role),
    side_profile(away_role),
    minutes[home_abbr],
    minutes[away_abbr],
    sim,
    rng,
    home_segments=schedules[home_abbr],
    away_segments=schedules[away_abbr],
)
shots = {home_abbr: allocated["home"], away_abbr: allocated["away"]}
for drawn in shots.values():
    if not drawn["metadata"]["segments_used"]:
        raise AssertionError("shots were not kept inside the Layer 3 segments")
for label, scenario in ((team_abbr, sim["team_pts"]), (opp_abbr, sim["opp_pts"])):
    target = np.floor(scenario.to_numpy(dtype=float) + 0.5).astype(np.int64)
    if not np.array_equal(shots[label]["PTS"].sum(axis=1), target):
        raise AssertionError(f"{label} points miss the scenario score")
for label, side in ((team_abbr, team_role), (opp_abbr, opp_role)):
    unavailable = side["available"].to_numpy() == 0
    if unavailable.any() and shots[label]["FGA"][:, unavailable].sum() != 0:
        raise AssertionError(f"{label} gave a shot to an unavailable player")


def shot_table(label, side, drawn):
    if not np.array_equal(drawn["FGA"].sum(axis=1), drawn["team_fga"]):
        raise AssertionError(f"{label} attempts do not sum to the shot pool")
    if not np.array_equal(drawn["TOV"].sum(axis=1), drawn["team_tov"]):
        raise AssertionError(f"{label} turnovers do not sum to the turnover pool")
    summary = pd.DataFrame(
        {
            "team": label,
            "player_name": side["player_name"].to_numpy(),
            "is_starter": side["is_starter"].to_numpy(),
            "fga_per_min": side_profile(side)["fga_per_min"].to_numpy(),
            "fga": drawn["FGA"].mean(axis=0),
            "pts": drawn["PTS"].mean(axis=0),
            "tov": drawn["TOV"].mean(axis=0),
            "pts_p10": np.percentile(drawn["PTS"], 10, axis=0),
            "pts_p90": np.percentile(drawn["PTS"], 90, axis=0),
        }
    )
    return summary.sort_values(["is_starter", "pts"], ascending=[False, False])


shown_shots = pd.concat(
    [
        shot_table(team_abbr, team_role, shots[team_abbr]),
        shot_table(opp_abbr, opp_role, shots[opp_abbr]),
    ],
    ignore_index=True,
)
attempt_means = {
    team_abbr: round(float(shots[team_abbr]["team_fga"].mean()), 1),
    opp_abbr: round(float(shots[opp_abbr]["team_fga"].mean()), 1),
}
print(f"live shots for {slate_day.date()}  mean team attempts {attempt_means}")
display(shown_shots.round(2))


/Users/alexgonzalez/Documents/nba_quant_analysis/nba_game_simulator/src/game_opportunities/train.py:38: UserWarning: zero-minute rows have positive statistics; per-minute denominators exclude them
  frame = _prepare_opportunity_logs(logs)
/Users/alexgonzalez/Documents/nba_quant_analysis/nba_game_simulator/src/game_opportunities/train.py:82: UserWarning: zero-minute rows have positive statistics; per-minute denominators exclude them
  frame = _prepare_opportunity_logs(logs)
/Users/alexgonzalez/Documents/nba_quant_analysis/nba_game_simulator/src/game_opportunities/train.py:180: UserWarning: zero-minute rows have positive statistics; per-minute denominators exclude them
  prepared = _prepare_opportunity_logs(logs)


shot model fit on 179,719 player-games through 2026-11-06  (179,719 rolling rows; each window stops before that game)
live shots for 2026-11-07  mean team attempts {'OKC': 91.5, 'WAS': 85.8}


,team,player_name,is_starter,fga_per_min,fga,pts,tov,pts_p10,pts_p90
0,OKC,Shai Gilgeous-Alexander,1.0,0.62,22.39,28.46,2.52,18.0,40.0
1,OKC,Ajay Mitchell,1.0,0.46,15.73,21.09,1.82,12.0,31.0
2,OKC,Alex Caruso,1.0,0.28,8.11,11.18,1.43,4.0,19.0
3,OKC,Isaiah Hartenstein,1.0,0.29,8.18,10.38,1.78,4.0,17.0
4,OKC,Kenrich Williams,1.0,0.32,8.08,10.30,1.04,4.0,18.0
5,OKC,Jared McCain,0.0,0.42,8.62,9.71,1.37,0.0,18.0
6,OKC,Aday Mara,0.0,0.31,5.14,6.86,1.35,0.0,14.0
7,OKC,Micah Peavy,0.0,0.28,4.47,5.58,0.57,0.0,13.0
8,OKC,Bennett Stirtz,0.0,0.31,3.96,5.48,0.54,0.0,13.0
9,OKC,Josh Dix,0.0,0.30,3.66,4.39,0.56,0.0,10.0


In [18]:
trained_events = training_events(history)
event_model = fit_events(history)
event_profile = live_events(
    history, players.drop(columns=["player_name", "team"])
)
event_profile["player_id"] = event_profile["player_id"].astype(
    players["player_id"].dtype
)
event_profile = names.drop(columns="available").merge(
    event_profile, on="player_id", how="left", validate="one_to_one"
)
print(
    f"event model fit through {event_model.cutoff}"
    f"  ({len(trained_events):,} rolling rows; each window stops before that game)"
)


def event_side(side):
    ordered = (
        event_profile.set_index("player_id").loc[side["player_id"]].reset_index()
    )
    if len(ordered) != len(side) or not ordered["player_id"].equals(side["player_id"]):
        raise AssertionError("event profile does not match the rotation")
    return ordered


availability = {
    team_abbr: team_role["available"].to_numpy() == 1,
    opp_abbr: opp_role["available"].to_numpy() == 1,
}
events, event_details = allocate_events(
    event_model,
    {team_abbr: event_side(team_role), opp_abbr: event_side(opp_role)},
    shots,
    minutes,
    rng,
    segments=schedules,
    availability=availability,
    return_details=True,
)
if not event_details["segments_used"]:
    raise AssertionError("events were not kept inside the Layer 3 segments")
for label, side in ((team_abbr, team_role), (opp_abbr, opp_role)):
    if event_details["sources"][label]["AST"] != "layer4":
        raise AssertionError(f"{label} assists were drawn again")
    if event_details["sources"][label]["OREB"] != "layer4":
        raise AssertionError(f"{label} offensive rebounds were drawn again")
    if not np.array_equal(events[label]["AST"], shots[label]["AST"]):
        raise AssertionError(f"{label} assists are not the Layer 4 assists")
    if not np.array_equal(events[label]["OREB"], shots[label]["OREB"]):
        raise AssertionError(f"{label} offensive rebounds are not the Layer 4 rebounds")
    if not np.array_equal(events[label]["REB"], events[label]["OREB"] + events[label]["DREB"]):
        raise AssertionError(f"{label} rebounds are not offensive plus defensive")
    unavailable = side["available"].to_numpy() == 0
    if unavailable.any():
        for stat in ("AST", "OREB", "DREB", "STL", "BLK"):
            if events[label][stat][:, unavailable].sum() != 0:
                raise AssertionError(f"{label} gave {stat} to an unavailable player")
for label, other in ((team_abbr, opp_abbr), (opp_abbr, team_abbr)):
    misses = shots[label]["FGA"].sum(axis=1) - shots[label]["FGM"].sum(axis=1)
    rebounds = events[label]["OREB"].sum(axis=1) + events[other]["DREB"].sum(axis=1)
    if np.any(rebounds > misses):
        raise AssertionError(f"rebounds on {label} misses exceed those misses")
    if np.any(events[other]["STL"].sum(axis=1) > shots[label]["TOV"].sum(axis=1)):
        raise AssertionError(f"steals exceed {label} turnovers")
    if np.any(events[other]["BLK"].sum(axis=1) > misses):
        raise AssertionError(f"blocks on {label} shots exceed its misses")


def event_table(label, side, drawn):
    summary = pd.DataFrame(
        {
            "team": label,
            "player_name": side["player_name"].to_numpy(),
            "is_starter": side["is_starter"].to_numpy(),
            "ast_per_min": event_side(side)["ast_per_min"].to_numpy(),
            "ast": drawn["AST"].mean(axis=0),
            "reb": drawn["REB"].mean(axis=0),
            "stl": drawn["STL"].mean(axis=0),
            "blk": drawn["BLK"].mean(axis=0),
        }
    )
    return summary.sort_values(["is_starter", "ast"], ascending=[False, False])


shown_events = pd.concat(
    [
        event_table(team_abbr, team_role, events[team_abbr]),
        event_table(opp_abbr, opp_role, events[opp_abbr]),
    ],
    ignore_index=True,
)
assist_means = {
    team_abbr: round(float(events[team_abbr]["AST"].sum(axis=1).mean()), 1),
    opp_abbr: round(float(events[opp_abbr]["AST"].sum(axis=1).mean()), 1),
}
print(f"live events for {slate_day.date()}  mean team assists {assist_means}")
display(shown_events.round(2))


event model fit through 2026-11-06  (179,719 rolling rows; each window stops before that game)
live events for 2026-11-07  mean team assists {'OKC': 27.1, 'WAS': 23.6}


,team,player_name,is_starter,ast_per_min,ast,reb,stl,blk
0,OKC,Shai Gilgeous-Alexander,1.0,0.15,4.73,4.85,1.40,1.12
1,OKC,Ajay Mitchell,1.0,0.13,4.22,4.87,1.61,0.55
2,OKC,Alex Caruso,1.0,0.14,4.10,6.12,1.25,0.52
3,OKC,Isaiah Hartenstein,1.0,0.11,3.53,8.97,1.25,0.52
4,OKC,Kenrich Williams,1.0,0.10,2.79,5.52,0.36,0.24
5,OKC,Jared McCain,0.0,0.07,1.83,3.11,0.85,0.24
6,OKC,Aday Mara,0.0,0.07,1.49,4.22,0.26,0.88
7,OKC,Bennett Stirtz,0.0,0.08,1.22,1.27,0.46,0.09
8,OKC,Micah Peavy,0.0,0.06,1.19,2.53,0.60,0.27
9,OKC,Josh Dix,0.0,0.05,0.93,1.59,0.40,0.20


In [19]:
box = assemble_box_score(
    sim,
    {"home": home_role["player_id"], "away": away_role["player_id"]},
    {"home": minutes[home_abbr], "away": minutes[away_abbr]},
    {"home": shots[home_abbr], "away": shots[away_abbr]},
    {"home": events[home_abbr], "away": events[away_abbr]},
    # Layer 4 already locked points with half-up rounding. Ties round differently here.
    check_score=False,
)
if box.n_sims != len(sim):
    raise AssertionError("the box score does not have one row per script")
for label, side_name in ((home_abbr, "home"), (away_abbr, "away")):
    stats = box.teams[side_name].stats
    if not np.array_equal(stats["PTS"], shots[label]["PTS"]):
        raise AssertionError(f"{label} box points are not the Layer 4 points")
    if not np.allclose(stats["MIN"], minutes[label]):
        raise AssertionError(f"{label} box minutes are not the rotation minutes")
    if not np.array_equal(stats["AST"], events[label]["AST"]):
        raise AssertionError(f"{label} box assists are not the Layer 5 assists")
    if not np.array_equal(stats["REB"], events[label]["REB"]):
        raise AssertionError(f"{label} box rebounds are not the Layer 5 rebounds")


def box_table(label, side, side_name):
    stats = box.teams[side_name].stats
    summary = pd.DataFrame(
        {
            "team": label,
            "player_id": side["player_id"].to_numpy(),
            "player_name": side["player_name"].to_numpy(),
            "is_starter": side["is_starter"].to_numpy(),
            "min": stats["MIN"].mean(axis=0),
            "pts": stats["PTS"].mean(axis=0),
            "fgm": stats["FGM"].mean(axis=0),
            "fga": stats["FGA"].mean(axis=0),
            "fg3m": stats["3PM"].mean(axis=0),
            "fg3a": stats["3PA"].mean(axis=0),
            "ftm": stats["FTM"].mean(axis=0),
            "fta": stats["FTA"].mean(axis=0),
            "reb": stats["REB"].mean(axis=0),
            "ast": stats["AST"].mean(axis=0),
            "stl": stats["STL"].mean(axis=0),
            "blk": stats["BLK"].mean(axis=0),
        }
    )
    return summary.sort_values(["is_starter", "pts"], ascending=[False, False])


shown_box = pd.concat(
    [
        box_table(home_abbr, home_role, "home"),
        box_table(away_abbr, away_role, "away"),
    ],
    ignore_index=True,
)
print(f"box score for {slate_day.date()}  {box.n_sims:,} rows, one game each")
display(shown_box.drop(columns="player_id").round(2))

played = pd.DataFrame(
    format_outcome(
        box,
        0,
        stats=(
            "MIN",
            "PTS",
            "REB",
            "AST",
            "STL",
            "BLK",
            "FGM",
            "FGA",
            "3PM",
            "3PA",
            "FTM",
            "FTA",
        ),
    )
)
played = played.rename(columns={"3PM": "FG3M", "3PA": "FG3A"})
lookup = pd.concat(
    [
        home_role.assign(side="home"),
        away_role.assign(side="away"),
    ],
    ignore_index=True,
)
lookup["player"] = lookup["player_id"].map(lambda value: str(int(value)))
lookup["team"] = lookup["side"]
lookup["team_name"] = lookup["side"].map({"home": home_abbr, "away": away_abbr})
played = played.merge(
    lookup[["team", "player", "player_name", "team_name"]],
    on=["team", "player"],
    how="left",
    validate="one_to_one",
)
played["team"] = played["team_name"]
played = played.loc[
    played["MIN"] > 0,
    [
        "team",
        "player_name",
        "MIN",
        "PTS",
        "REB",
        "AST",
        "STL",
        "BLK",
        "FGM",
        "FGA",
        "FG3M",
        "FG3A",
        "FTM",
        "FTA",
    ],
]
print(f"simulation 0 for {slate_day.date()}")
display(played.reset_index(drop=True))

leaders = (
    shown_box.loc[shown_box["team"].eq(team_abbr)]
    .sort_values(["is_starter", "pts"], ascending=[False, False])
    .head(2)
)
slate_side = "home" if team_abbr == home_abbr else "away"
props = []
for row in leaders.itertuples(index=False):
    index = list(box.teams[slate_side].player_ids).index(str(int(row.player_id)))
    column = box.teams[slate_side].stats["PTS"][:, index]
    props.append(Prop(slate_side, row.player_id, "PTS", float(np.floor(np.median(column)) + 0.5)))
priced = prop_probability(box, props)
print(
    f"P({leaders.iloc[0].player_name} over {props[0].line} and "
    f"{leaders.iloc[1].player_name} over {props[1].line})"
    f" = {priced.joint:.3f}; product of the separate chances = {priced.independent_product:.3f}"
)
print(f"separate chances = {tuple(round(value, 3) for value in priced.marginals)}")


box score for 2026-11-07  5,000 rows, one game each


,team,player_name,is_starter,min,pts,fgm,fga,fg3m,fg3a,ftm,fta,reb,ast,stl,blk
0,OKC,Shai Gilgeous-Alexander,1.0,36.26,28.46,10.47,22.39,2.70,7.10,4.82,5.87,4.85,4.73,1.40,1.12
1,OKC,Ajay Mitchell,1.0,34.21,21.09,8.25,15.73,1.43,4.24,3.16,3.93,4.87,4.22,1.61,0.55
2,OKC,Alex Caruso,1.0,29.05,11.18,3.94,8.11,1.90,4.60,1.40,1.74,6.12,4.10,1.25,0.52
3,OKC,Isaiah Hartenstein,1.0,28.14,10.38,4.09,8.18,0.50,1.42,1.70,2.09,8.97,3.53,1.25,0.52
4,OKC,Kenrich Williams,1.0,25.77,10.30,3.75,8.08,1.71,4.24,1.08,1.41,5.52,2.79,0.36,0.24
5,OKC,Jared McCain,0.0,20.50,9.71,3.65,8.62,1.32,4.12,1.10,1.40,3.11,1.83,0.85,0.24
6,OKC,Aday Mara,0.0,16.89,6.86,2.72,5.14,0.36,0.97,1.07,1.47,4.22,1.49,0.26,0.88
7,OKC,Micah Peavy,0.0,15.57,5.58,2.05,4.47,0.70,1.77,0.78,0.95,2.53,1.19,0.60,0.27
8,OKC,Bennett Stirtz,0.0,12.67,5.48,1.92,3.96,0.93,2.17,0.71,0.88,1.27,1.22,0.46,0.09
9,OKC,Josh Dix,0.0,12.20,4.39,1.66,3.66,0.58,1.69,0.50,0.63,1.59,0.93,0.40,0.20


simulation 0 for 2026-11-07


,team,player_name,MIN,PTS,REB,AST,STL,BLK,FGM,FGA,FG3M,FG3A,FTM,FTA
0,OKC,Shai Gilgeous-Alexander,39.014278,29,8,3,1,1,11,25,3,10,4,6
1,OKC,Ajay Mitchell,21.609961,11,6,1,0,0,3,8,1,2,4,5
2,OKC,Isaiah Hartenstein,29.580377,9,11,4,0,0,4,12,1,2,0,0
3,OKC,Kenrich Williams,17.858571,9,1,2,0,2,3,10,3,9,0,2
4,OKC,Alex Caruso,40.820060,12,8,3,1,0,4,11,1,5,3,3
5,OKC,Jared McCain,23.624926,11,2,3,0,0,4,8,0,0,3,4
6,OKC,Micah Peavy,22.065204,6,4,2,1,0,2,4,0,1,2,4
7,OKC,Aday Mara,20.612874,10,5,1,0,0,3,5,1,1,3,4
8,OKC,Josh Dix,24.813749,5,3,1,1,0,2,4,0,2,1,1
9,WAS,Trae Young,31.896926,19,4,5,1,0,5,10,3,4,6,7


P(Shai Gilgeous-Alexander over 28.5 and Ajay Mitchell over 21.5) = 0.209; product of the separate chances = 0.223
separate chances = (0.481, 0.463)


In [20]:
BOOK = "novig"

from game_boxscore import portfolio_risk, price_over, stat_values
from src.live.prop_markets import attach_roster, book_names, latest_prop_markets

markets, market_audit = latest_prop_markets(REPO / "data" / "props", BOOK)
others = ", ".join(name for name in book_names() if name != market_audit["book"])
print(
    f"Layer 7 {market_audit['book']} from {market_audit['source_file']}. "
    f"Change BOOK to {others}."
)
if market_audit["book"] == "prizepicks":
    print("PrizePicks has no two-way price. Both sides are -119.")

roster = pd.concat(
    [
        home_role.assign(team=home_abbr, side="home"),
        away_role.assign(team=away_abbr, side="away"),
    ],
    ignore_index=True,
)
offered = attach_roster(
    markets,
    roster.loc[:, ["team", "side", "player_id", "player_name"]],
)
slate_dates = sorted(set(pd.to_datetime(offered["slate_date"]).dt.date))
if offered.empty:
    print(f"no {market_audit['book']} lines matched this game")
elif slate_dates != [slate_day.date()]:
    print(f"market dates {slate_dates}; the slate date is {slate_day.date()}")


def stat_board(stat, label):
    rows = []
    ranked = []
    for offer in offered.loc[offered["stat"].eq(stat)].itertuples(index=False):
        index = list(box.teams[offer.side].player_ids).index(offer.player_id)
        minutes = box.teams[offer.side].stats["MIN"][:, index]
        column = stat_values(box, offer.side, stat)[:, index]
        priced = price_over(
            box,
            Prop(offer.side, offer.player_id, stat, float(offer.line)),
            float(offer.over_american),
        )
        rows.append(
            {
                "team": offer.team,
                "player_name": offer.player_name,
                "min": float(minutes.mean()),
                label: float(column.mean()),
                f"{label}_line": float(offer.line),
                "odds": int(offer.over_american),
                "prop_over": priced.p_over,
                "over_ev": priced.ev,
            }
        )
        ranked.append(
            {
                "team": offer.team,
                "player_name": offer.player_name,
                "stat": label,
                "min": float(minutes.mean()),
                "line": float(offer.line),
                "odds": int(offer.over_american),
                "prop_over": priced.p_over,
                "over_ev": priced.ev,
                "side": offer.side,
                "player_id": offer.player_id,
                "box_stat": stat,
            }
        )
    board = pd.DataFrame(rows)
    if not board.empty:
        board = board.sort_values("over_ev", ascending=False)
    return board, pd.DataFrame(ranked)


ranked_rows = []
for stat, label in (
    ("PTS", "pts"),
    ("REB", "reb"),
    ("AST", "ast"),
    ("STL", "stl"),
    ("BLK", "blk"),
    ("PA", "p+a"),
    ("PRA", "p+r+a"),
    ("RA", "r+a"),
):
    board, ranked = stat_board(stat, label)
    if not ranked.empty:
        ranked_rows.append(ranked)
    print(f"{label} overs for {slate_day.date()}")
    if board.empty:
        print(f"no {label} line for this game")
        continue
    shown = board.copy()
    shown["min"] = shown["min"].round(2)
    shown[label] = shown[label].round(2)
    shown["prop_over"] = shown["prop_over"].round(3)
    shown["over_ev"] = shown["over_ev"].round(3)
    display(shown.reset_index(drop=True))

if not ranked_rows:
    print("no priced props for this game")
else:
    ranked = pd.concat(ranked_rows, ignore_index=True)
    top = ranked.sort_values("over_ev", ascending=False).head(5).reset_index(drop=True)
    print(f"top {len(top)} overs by expected value")
    shown_top = top.drop(columns=["side", "player_id", "box_stat"]).copy()
    shown_top["min"] = shown_top["min"].round(2)
    shown_top["prop_over"] = shown_top["prop_over"].round(3)
    shown_top["over_ev"] = shown_top["over_ev"].round(3)
    display(shown_top)
    legs = [
        Prop(row.side, row.player_id, row.box_stat, float(row.line))
        for row in top.itertuples(index=False)
    ]
    held = portfolio_risk(box, legs, top["odds"].tolist())
    print(
        f"portfolio of these {held.n}, $1 on each over. "
        f"expected profit = {held.expected_profit:.3f}; "
        f"risk = {held.risk:.3f}; "
        f"P(profit < 0) = {held.p_loss:.3f}; "
        f"P(all clear) = {held.joint:.3f}; "
        f"product of the separate chances = {held.independent_product:.3f}"
    )


Layer 7 novig from novig_nba_2026-11-07_111501_props.json. Change BOOK to prizepicks, underdog, prophetx, pinnacle, fanduel, draftkings.
pts overs for 2026-11-07


,team,player_name,min,pts,pts_line,odds,prop_over,over_ev
0,OKC,Alex Caruso,29.05,11.18,7.5,-111,0.706,0.342
1,OKC,Isaiah Hartenstein,28.14,10.38,8.5,102,0.615,0.243
2,OKC,Kenrich Williams,25.77,10.30,7.5,-130,0.655,0.158
3,OKC,Ajay Mitchell,34.21,21.09,19.5,-100,0.569,0.137
4,WAS,Kyshawn George,31.35,15.16,15.5,-102,0.459,-0.091
5,WAS,Bilal Coulibaly,18.66,6.61,7.5,-106,0.402,-0.219
6,OKC,Aday Mara,16.89,6.86,8.5,104,0.350,-0.286
7,OKC,Shai Gilgeous-Alexander,36.26,28.46,31.5,102,0.350,-0.293
8,WAS,Trae Young,33.38,16.61,18.5,-115,0.369,-0.310
9,OKC,Jared McCain,20.50,9.71,13.5,-100,0.265,-0.469


reb overs for 2026-11-07


,team,player_name,min,reb,reb_line,odds,prop_over,over_ev
0,OKC,Alex Caruso,29.05,6.12,3.5,106,0.744,0.533
1,WAS,AJ Dybantsa,31.30,5.93,5.5,122,0.515,0.142
2,WAS,Trae Young,33.38,3.67,2.5,-133,0.647,0.133
3,OKC,Ajay Mitchell,34.21,4.87,3.5,-133,0.646,0.132
4,OKC,Kenrich Williams,25.77,5.52,4.5,-104,0.565,0.108
5,OKC,Shai Gilgeous-Alexander,36.26,4.85,3.5,-182,0.645,0.000
6,WAS,Anthony Davis,32.71,9.10,8.5,-144,0.512,-0.133
7,WAS,Bilal Coulibaly,18.66,2.96,2.5,-160,0.524,-0.148
8,OKC,Jared McCain,20.50,3.11,3.5,117,0.381,-0.172
9,WAS,Kyshawn George,31.35,4.32,4.5,-144,0.422,-0.285


ast overs for 2026-11-07


,team,player_name,min,ast,ast_line,odds,prop_over,over_ev
0,OKC,Alex Caruso,29.05,4.10,2.5,122,0.739,0.640
1,OKC,Isaiah Hartenstein,28.14,3.53,3.5,122,0.466,0.034
2,WAS,AJ Dybantsa,31.30,2.37,1.5,-190,0.676,0.032
3,OKC,Ajay Mitchell,34.21,4.22,3.5,-160,0.601,-0.024
4,OKC,Jared McCain,20.50,1.83,1.5,-135,0.526,-0.084
5,WAS,Kyshawn George,31.35,3.62,4.5,113,0.296,-0.369
6,OKC,Shai Gilgeous-Alexander,36.26,4.73,5.5,-122,0.341,-0.380
7,WAS,Trae Young,33.38,5.94,8.5,102,0.166,-0.665


... (7 outputs left out)

,team,player_name,min,r+a,r+a_line,odds,prop_over,over_ev
0,OKC,Isaiah Hartenstein,28.14,12.50,13.5,-122,0.393,-0.284
1,OKC,Shai Gilgeous-Alexander,36.26,9.58,10.5,-144,0.376,-0.363
2,WAS,Trae Young,33.38,9.61,11.5,-120,0.282,-0.483


top 5 overs by expected value


,team,player_name,stat,min,line,odds,prop_over,over_ev
0,OKC,Alex Caruso,ast,29.05,2.5,122,0.739,0.640
1,OKC,Alex Caruso,p+r+a,29.05,13.5,-113,0.823,0.551
2,OKC,Alex Caruso,reb,29.05,3.5,106,0.744,0.533
3,OKC,Alex Caruso,pts,29.05,7.5,-111,0.706,0.342
4,OKC,Ajay Mitchell,stl,34.21,1.5,178,0.471,0.308


portfolio of these 5, $1 on each over. expected profit = 2.375; risk = 2.692; P(profit < 0) = 0.173; P(all clear) = 0.202; product of the separate chances = 0.150


In [21]:
from itertools import combinations

from game_boxscore import Prop, prop_probability

positive = ranked.loc[ranked["over_ev"] > 0].reset_index(drop=True)
pairs = []
for i, j in combinations(range(len(positive)), 2):
    a = positive.iloc[i]
    b = positive.iloc[j]
    priced = prop_probability(
        box,
        [
            Prop(a.side, a.player_id, a.box_stat, float(a.line)),
            Prop(b.side, b.player_id, b.box_stat, float(b.line)),
        ],
    )
    if a.player_id == b.player_id:
        link = "same player"
    elif a.team == b.team:
        link = "teammates"
    else:
        link = "opponents"
    pairs.append(
        {
            "link": link,
            "a": f"{a.player_name} {a.stat} {a.line:g}",
            "b": f"{b.player_name} {b.stat} {b.line:g}",
            "p_a": priced.marginals[0],
            "p_b": priced.marginals[1],
            "joint": priced.joint,
            "product": priced.independent_product,
            "gap": priced.joint - priced.independent_product,
        }
    )

board = pd.DataFrame(pairs).sort_values("gap", ascending=False).reset_index(drop=True)
print(
    f"correlation pairs among {len(positive)} overs with over_ev > 0, "
    f"{market_audit['book']}, {slate_day.date()}, {box.n_sims:,} rows"
)
shown = board.copy()
for column in ("p_a", "p_b", "joint", "product", "gap"):
    shown[column] = shown[column].round(3)
display(shown)
print(
    "top row is the parlay: they clear together more often than the product. "
    "bottom row is the pair of singles."
)

correlation pairs among 19 overs with over_ev > 0, novig, 2026-11-07, 5,000 rows


,link,a,b,p_a,p_b,joint,product,gap
0,same player,Ajay Mitchell pts 19.5,Ajay Mitchell p+a 23.5,0.569,0.575,0.531,0.327,0.204
1,same player,Ajay Mitchell p+a 23.5,Ajay Mitchell p+r+a 27.5,0.575,0.606,0.543,0.349,0.195
2,same player,Ajay Mitchell pts 19.5,Ajay Mitchell p+r+a 27.5,0.569,0.606,0.524,0.345,0.179
3,same player,Alex Caruso pts 7.5,Alex Caruso p+r+a 13.5,0.706,0.823,0.684,0.581,0.103
4,same player,Ajay Mitchell reb 3.5,Ajay Mitchell p+r+a 27.5,0.646,0.606,0.468,0.392,0.076
...,...,...,...,...,...,...,...,...
166,teammates,Kenrich Williams pts 7.5,Ajay Mitchell reb 3.5,0.655,0.646,0.414,0.423,-0.009
167,teammates,Isaiah Hartenstein pts 8.5,Kenrich Williams pts 7.5,0.615,0.655,0.393,0.403,-0.009
168,teammates,Isaiah Hartenstein pts 8.5,Ajay Mitchell p+a 23.5,0.615,0.575,0.344,0.354,-0.010
169,teammates,Ajay Mitchell pts 19.5,Isaiah Hartenstein pts 8.5,0.569,0.615,0.339,0.350,-0.011


top row is the parlay: they clear together more often than the product. bottom row is the pair of singles.
